In [1]:
#!/usr/bin/env python
# coding: utf-8
###
###  Purpose and Construction 
###
###  We use the function UCTSEARCH to perform the MCTS.
###  UCTSEARCH performs the expansion, the simulation, and the back-propagation.
###  Those three phases are executed by functions 
###  whose name might not correctly suggest their roles.
###  
###  TREEPOLICY performs the expansion, 
###  DEFAULTPOLICY the simulation,
###  BACK the back-propagation.
###
###  A search is made of leaves (or nodes).
###  To build a tree, we use two classes (Node and State).
###  The Node class represents a node and performs various tasks to build a tree.
###  A node has its state that represents the data of the circuit.
###  The class State represents the state of the circuit and modifies it. 
###  
###  The comments embedded in the following source code show you the details
###  regarding the task flow and the roles of the function and the classes.    
###  
import logging, logging.handlers, pathlib
import multiprocessing
logger = logging.getLogger(__name__) # NOTSET (0)
def jisaku_initializer(que):
    """各子プロセスで最初に 1 回だけ実行される関数です。"""
    root = logging.getLogger() # 各子プロセスのルートロガー
    root.setLevel(logging.INFO) # WARNING (30) ⇒ DEBUG (10)
    qh = logging.handlers.QueueHandler(que) # NOTSET (0)
    root.addHandler(qh)
    #root.info('(キューハンドラ追加完了)')
    #root.info(f'root.level: {logging.getLevelName(root.level)} ({root.level})')
    #root.info(f'logger.level: {logging.getLevelName(logger.level)} ({logger.level})')
    #root.info(f'qh.level: {logging.getLevelName(qh.level)} ({qh.level})')
    return

import multiprocessing
import threading
import numpy as np

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import CommutativeCancellation, Optimize1qGates
from qiskit import QuantumCircuit
from qiskit import QuantumCircuit, transpile
from qiskit.synthesis import generate_basic_approximations
from qiskit.transpiler.passes import SolovayKitaev
from qiskit.quantum_info import Operator
from qiskit import QuantumCircuit, transpile
from qiskit.synthesis import generate_basic_approximations
from qiskit.transpiler.passes import SolovayKitaev
from qiskit.quantum_info import Operator
from qiskit.circuit.quantumcircuit import QuantumCircuit
from qiskit.dagcircuit import DAGDependency
from qiskit.converters.circuit_to_dagdependency import circuit_to_dagdependency
from qiskit.converters.dagdependency_to_circuit import dagdependency_to_circuit
from qiskit.converters.dag_to_dagdependency import dag_to_dagdependency
from qiskit.converters.dagdependency_to_dag import dagdependency_to_dag
from qiskit.transpiler.basepasses import TransformationPass
from qiskit.circuit.library.templates import template_nct_2a_1, template_nct_2a_2, template_nct_2a_3
from qiskit.quantum_info.operators.operator import Operator
from qiskit.transpiler.exceptions import TranspilerError
from qiskit.transpiler.passes.optimization.template_matching import (
 TemplateMatching,
 TemplateSubstitution,
 MaximalMatches,
)
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit_aer import Aer
import sys
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler.passes import CommutativeCancellation, Optimize1qGates
from qiskit.transpiler import generate_preset_pass_manager
from collections.abc import Callable, Iterator
from typing import overload
import qiskit.circuit.library.templates.clifford as clifford_template_module
import qiskit.circuit.library.templates.nct as nct_template_module
from qiskit.passmanager import DoWhileController
from qiskit.passmanager.base_tasks import Task
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Depth, FixedPoint, Size, TemplateOptimization
import qiskit.quantum_info as qi
import random
import itertools
import copy

###
### We read the data of a circuit from a qasm file.
###
#circuit=QuantumCircuit.from_qasm_file("../circuit-optimization-experiment-shuhei-main/_CtrlSubNoCarry_10.qasm")


###
### We construct the circuit with gates "x", "ccx", "cx", and "h". 
### "h" might not be there.
###
#qc=circuit.decompose().decompose().decompose()
#qc_transpiled = transpile(circuit, basis_gates=["x", "ccx","cx","h"])
#print(qc_transpiled.draw())
#print(qc_transpiled.count_ops())




###
### We prepare two lists of template circuits. We use NCT templates hereafter.
###
CLIFFORD_TEMPLATES = {
    name: clifford_template_module.__dict__[name]
    for name in dir(clifford_template_module)
    if not (
        name.startswith("_") or name == "clifford_6_4"
 ) # Exclude clifford_6_4 as it is not a valid template
}

NCT_TEMPLATES = {
    name: nct_template_module.__dict__[name]
    for name in dir(nct_template_module)
    if name[0] != "_"
}
ALL_TEMPLATES = {**CLIFFORD_TEMPLATES, **NCT_TEMPLATES}


###
### We check the validity of templates. 
### Usually, they shall pass this check.
###
for ky in NCT_TEMPLATES.keys():
    tc=NCT_TEMPLATES[ky]()
    try:
        pass_manager = generate_preset_pass_manager(
        basis_gates=["x","cx","ccx","h","cz"],
 )
        op = qi.Operator(tc)
        if np.linalg.det(op.to_matrix())!=1:
            print("ERROR")
    except Exception as e:
        #print(e)
        1


###
###  We prepare cost dictionaries.
###  The weights for x, cx, and ccx gates are negative.
###  If we use this dictionary in the template optimization,
###  the optimization goes in the inverse direction with the growth of the complexity.
###  But this dictionary is not used currently.
###
import qiskit.transpiler.passes.optimization.template_matching.template_substitution as tempsub
mycost_dict=tempsub.TemplateSubstitution(None,None,None).cost_dict
for ky in mycost_dict:
    mycost_dict[ky]*=-1

###
### A function to perform the optimization up to the fixed-point.
###

def _construct_repeat_to_fixed_point_pass(
    base_task: Task | list[Task],
) -> DoWhileController:
    if not isinstance(base_task, list):
        base_task = [base_task]
    checks: list[Task] = [Depth(recurse=True), FixedPoint("depth")]
    checks += [Size(recurse=True), FixedPoint("size")]

    def _opt_control(property_set):
        return (not property_set["depth_fixed_point"]) or (
            not property_set["size_fixed_point"]
 )

    return DoWhileController(base_task + checks, do_while=_opt_control)

#
# Usage
#
#p=_construct_repeat_to_fixed_point_pass(TemplateOptimization(template_listN)  )  
#pass_manager = PassManager()
#pass_manager.append(p)
#optimized_qc = pass_manager.run(qc_transpiled)
##print(optimized_qc.depth(),optimized_qc.count_ops())

##
## We check the standard cost dictionary used in the Template Optimization.
##

TO=TemplateSubstitution(None,None,None)
#print()
#print("Default cost dictionary is as follows.")
#print(TO.cost_dict)


###
### We prepare a set of cost dictionaries : a list mydictionaries.
### They have variety of weights for x, cx, and ccx,
### which are positive, zero, or negative.
###
#print()
#print("Now I prepare cost dictionaries for the later usage: they shall alter the complexity of the cicuit after the optimization.")
#print("They are kept in two lists: mydictionaries (causing large changes) and mydictionariesB (causing small changes).")
#print("Hereafter I use the former.")
my_cost_dict=TO.cost_dict
import itertools
import copy
directprd=itertools.product(range(-1,2),range(-1,2),range(-1,2) )
mydictionaries=[]
for u0,u1,u2 in directprd:
    if u0!=0 or u1!=0 or u2!=0:
        dictdummy=copy.deepcopy(TO.cost_dict)
        dictdummy['x']*=u0
        dictdummy['cx']*=u1
        dictdummy['ccx']*=u2
        mydictionaries.append(dictdummy)

###
### We prepare another set of custom cost dictionaries.
### They have variety of weights for x, cx, and ccx,
### which are positive, zero.
### We prepare them with the expectation that the perturbations caused by them are milder.
###
directprd=itertools.product(range(0,2),range(0,2),range(0,2) )
mydictionariesB=[]
for u0,u1,u2 in directprd:
    dictdummy=copy.deepcopy(TO.cost_dict)
    dictdummy['x']*=u0
    dictdummy['cx']*=u1
    dictdummy['ccx']*=u2
    mydictionariesB.append(dictdummy)


###
###  qc_transpiled is the design of the circuit that we optimize.
###  It is copied to optimized_qc.
###
pass_manager = PassManager()
template_listC=[CLIFFORD_TEMPLATES[ky]() for ky in CLIFFORD_TEMPLATES.keys()]
template_listN=[NCT_TEMPLATES[ky]() for ky in NCT_TEMPLATES.keys()]
#optimized_qc=copy.deepcopy(qc_transpiled)

#for l,u in enumerate(template_listN+template_listC):
#    print(l)
#    print(u)

In [2]:
template_listN[8].draw()

q_0: ──■────■────■────■────■──
       │  ┌─┴─┐  │  ┌─┴─┐  │  
q_1: ──■──┤ X ├──■──┤ X ├──┼──
     ┌─┴─┐└───┘┌─┴─┐└───┘┌─┴─┐
q_2: ┤ X ├─────┤ X ├─────┤ X ├
     └───┘     └───┘     └───┘

In [3]:
ideal I = ccx012*cx21*cx02*ccx012*cx01*ccx012*cx21*cx01*ccx012-1;
ideal I1= ccx012*ccx012-1,cx21*cx21-1,cx01*cx01-1,cx02*cx02-1,cx01*cx02-cx02*cx01;
ideal J = letplaceGBasis(I+I1);
poly Q1=ccx012*cx01*ccx012*cx01*cx02;
reduce(Q1,J);

SyntaxError: invalid syntax (3813658770.py, line 1)

In [4]:
template_listN[26].draw()

q_0: ──■─────────■────■────■────■─────────■────■──
     ┌─┴─┐     ┌─┴─┐┌─┴─┐  │  ┌─┴─┐       │  ┌─┴─┐
q_1: ┤ X ├──■──┤ X ├┤ X ├──┼──┤ X ├──■────┼──┤ X ├
     └─┬─┘┌─┴─┐└───┘└─┬─┘┌─┴─┐└─┬─┘┌─┴─┐┌─┴─┐└─┬─┘
q_2: ──■──┤ X ├───────■──┤ X ├──■──┤ X ├┤ X ├──■──
          └───┘          └───┘     └───┘└───┘

In [5]:
from qiskit import qasm2
def qctomonom(qc):
    qcstr=qasm2.dumps(qc).split("\n")
    w=""
    for u in qcstr[3:]:
        w+=u.replace("q[","Q[").replace(";","").replace(",","_").replace(" ","_").replace("[","").replace("]","")+"*"
    w=w[:-1]
    return w,list(set(w.split("*")))
def rotate(ringstring0,A,B):
    ringstring=copy.deepcopy(ringstring0)
    for u,v in zip(A,B):
        ringstring=ringstring.replace("Q"+str(u),"P"+str(v))
    return ringstring.replace("P","Q")

In [6]:
f1,f2=qctomonom(template_listN[8])
f1.replace("_Q","")=="ccx012*cx01*ccx012*cx01*cx02"

True

In [7]:
ideal I = ccx012*cx21*cx02*ccx012*cx01*ccx012*cx21*cx01*ccx012-1;
ideal I1= ccx012*ccx012-1,cx21*cx21-1,cx01*cx01-1,cx02*cx02-1,cx01*cx02-cx02*cx01;
ideal J = letplaceGBasis(I+I1);
poly Q1="ccx012*cx01*ccx012*cx01*cx02"
reduce(Q1,J);

SyntaxError: invalid syntax (2614666536.py, line 1)

In [8]:
g1,g2=qctomonom(template_listN[26])
print(g1.replace("_Q","")=="ccx012*cx21*cx02*ccx012*cx01*ccx012*cx21*cx01*ccx012")
g1r=rotate(g1,[0,1,2],[0,2,1])
print(g1r.replace("_Q","")=="ccx012*cx21*cx02*ccx012*cx01*ccx012*cx21*cx01*ccx012")

False
True


In [9]:
MONOMS=g1r.split("*")+f1.split("*")
MONOMS=list(set(MONOMS))
MONOMS.sort()
print(MONOMS)
MONOMS=[u.replace("_Q","") for u in MONOMS]
MONOMS.sort()
MONOMS

['ccx_Q0_Q1_Q2', 'cx_Q0_Q1', 'cx_Q0_Q2', 'cx_Q2_Q1']


['ccx012', 'cx01', 'cx02', 'cx21']

In [10]:
LIB "freegb.lib";
ring r=0,(ccx012,cx21,cx01,cx02),dp;
def R = freeAlgebra(r, 10);

setring R;

SyntaxError: invalid syntax (2365145933.py, line 1)

In [11]:
v=rotate(f1,[0,1,2],[0,2,1])
v.replace("_Q","")

'ccx021*cx02*ccx021*cx02*cx01'

In [12]:
qc=template_listN[3]
def CircuitStr(qc):
    circuitstr=''
    for instruction in qc.data:
        gate_name = instruction.operation.name
        qubit_objects = instruction.qubits
        
        # Get the global integer index for each qubit involved
        qubit_indices = [qc.find_bit(q).index for q in qubit_objects]
        
        #print(f"Gate: {gate_name}, Qubit indices: {qubit_indices}")
        gatestr=gate_name
        for u in qubit_indices:
            gatestr+="_"+str(u)
        #print(gatestr)
        circuitstr+=gatestr+"*"
    circuitstr=circuitstr[0:-1]
    #print(circuitstr)
    return circuitstr+"-1"


def GetMonomials(expr,MONOMS=[]):
    MONOMS.extend(expr.replace("*", " ").replace("-"," ").split())
    return MONOMS
    
EQS=[CircuitStr(u) for u in template_listN]

MONOMS=[]
for u in EQS:
    GetMonomials(u,MONOMS)
MONOMS=list(set(MONOMS))
MONOMS.sort()
print(MONOMS)
print(EQS)

['1', 'ccx_0_1_2', 'ccx_0_1_3', 'ccx_0_2_1', 'ccx_0_2_3', 'ccx_1_2_0', 'ccx_1_2_3', 'ccx_1_2_4', 'cx_0_1', 'cx_0_2', 'cx_1_0', 'cx_1_2', 'cx_2_1', 'x_0', 'x_1', 'x_2']
['x_0*x_0-1', 'cx_0_1*cx_0_1-1', 'ccx_0_1_2*ccx_0_1_2-1', 'ccx_1_2_4*ccx_0_2_3*ccx_1_2_4*ccx_0_2_3-1', 'ccx_0_1_3*cx_1_2*ccx_0_1_3*cx_1_2-1', 'cx_0_2*cx_0_1*cx_0_2*cx_0_1-1', 'ccx_1_2_3*ccx_0_2_3*ccx_1_2_3*ccx_0_2_3-1', 'ccx_0_1_2*cx_1_2*ccx_0_1_2*cx_1_2-1', 'ccx_0_1_2*cx_0_1*ccx_0_1_2*cx_0_1*cx_0_2-1', 'ccx_0_1_2*x_1*ccx_0_1_2*x_1*cx_0_2-1', 'cx_1_2*cx_0_1*cx_1_2*cx_0_1*cx_0_2-1', 'cx_0_1*x_0*cx_0_1*x_0*x_1-1', 'cx_0_1*cx_1_0*cx_0_1*cx_1_0*cx_0_1*cx_1_0-1', 'ccx_0_1_2*ccx_0_2_1*ccx_0_1_2*ccx_0_2_1*ccx_0_1_2*ccx_0_2_1-1', 'cx_1_2*ccx_0_2_1*cx_1_2*ccx_0_2_1*ccx_0_1_2*ccx_0_2_1-1', 'cx_1_2*ccx_0_2_1*cx_1_2*cx_2_1*ccx_0_1_2*cx_2_1-1', 'ccx_0_1_2*cx_1_2*ccx_0_2_1*ccx_0_1_2*cx_1_2*ccx_0_2_1-1', 'cx_1_2*ccx_0_1_2*ccx_0_2_1*cx_1_2*ccx_0_1_2*ccx_0_2_1-1', 'ccx_0_1_2*cx_2_1*ccx_0_1_2*cx_2_1*ccx_0_1_2*ccx_0_2_1-1', 'cx_1_0*cx_0_1*

In [13]:
for u in template_listN[3].data:
    print(u.operation.name,u.qubits)

ccx (<Qubit register=(5, "q"), index=1>, <Qubit register=(5, "q"), index=2>, <Qubit register=(5, "q"), index=4>)
ccx (<Qubit register=(5, "q"), index=0>, <Qubit register=(5, "q"), index=2>, <Qubit register=(5, "q"), index=3>)
ccx (<Qubit register=(5, "q"), index=1>, <Qubit register=(5, "q"), index=2>, <Qubit register=(5, "q"), index=4>)
ccx (<Qubit register=(5, "q"), index=0>, <Qubit register=(5, "q"), index=2>, <Qubit register=(5, "q"), index=3>)


In [14]:
template_listN[6].draw(output="latex_source")

'\\documentclass[border=2px]{standalone}\n\n\\usepackage[braket, qm]{qcircuit}\n\\usepackage{graphicx}\n\n\\begin{document}\n\\scalebox{1.0}{\n\\Qcircuit @C=1.0em @R=0.8em @!R { \\\\\n\t \t\\nghost{{q}_{0} :  } & \\lstick{{q}_{0} :  } & \\qw & \\ctrl{2} & \\qw & \\ctrl{2} & \\qw & \\qw\\\\\n\t \t\\nghost{{q}_{1} :  } & \\lstick{{q}_{1} :  } & \\ctrl{1} & \\qw & \\ctrl{1} & \\qw & \\qw & \\qw\\\\\n\t \t\\nghost{{q}_{2} :  } & \\lstick{{q}_{2} :  } & \\ctrl{1} & \\ctrl{1} & \\ctrl{1} & \\ctrl{1} & \\qw & \\qw\\\\\n\t \t\\nghost{{q}_{3} :  } & \\lstick{{q}_{3} :  } & \\targ & \\targ & \\targ & \\targ & \\qw & \\qw\\\\\n\\\\ }}\n\\end{document}'

In [15]:
tc=template_listN[6]

In [16]:
tc.num_qubits

4

In [17]:
QC=QuantumCircuit(4)

In [18]:
QC=QuantumCircuit(4)
tc.to_gate()
QC.append(tc.to_gate(),[0,1,2,3])
QC.append(tc.to_gate(),[0,1,3,2])
QC.decompose().draw()

q_0: ───────■─────────■─────────■─────────■──
            │         │         │         │  
q_1: ──■────┼────■────┼────■────┼────■────┼──
       │    │    │    │  ┌─┴─┐┌─┴─┐┌─┴─┐┌─┴─┐
q_2: ──■────■────■────■──┤ X ├┤ X ├┤ X ├┤ X ├
     ┌─┴─┐┌─┴─┐┌─┴─┐┌─┴─┐└─┬─┘└─┬─┘└─┬─┘└─┬─┘
q_3: ┤ X ├┤ X ├┤ X ├┤ X ├──■────■────■────■──
     └───┘└───┘└───┘└───┘

In [10]:
!pip install pdftocairo

ERROR: Could not find a version that satisfies the requirement pdftocairo (from versions: none)
ERROR: No matching distribution found for pdftocairo


In [11]:
print(template_listN[26].draw(output="latex_source"))

\documentclass[border=2px]{standalone}

\usepackage[braket, qm]{qcircuit}
\usepackage{graphicx}

\begin{document}
\scalebox{1.0}{
\Qcircuit @C=1.0em @R=0.8em @!R { \\
	 	\nghost{{q}_{0} :  } & \lstick{{q}_{0} :  } & \ctrl{1} & \qw & \ctrl{1} & \ctrl{1} & \ctrl{2} & \ctrl{1} & \qw & \ctrl{2} & \ctrl{1} & \qw & \qw\\
	 	\nghost{{q}_{1} :  } & \lstick{{q}_{1} :  } & \targ & \ctrl{1} & \targ & \targ & \qw & \targ & \ctrl{1} & \qw & \targ & \qw & \qw\\
	 	\nghost{{q}_{2} :  } & \lstick{{q}_{2} :  } & \ctrl{-1} & \targ & \qw & \ctrl{-1} & \targ & \ctrl{-1} & \targ & \targ & \ctrl{-1} & \qw & \qw\\
\\ }}
\end{document}


In [ ]:
for jdx,u in enumerate(template_listN):
    #if jdx!=8:
    #    continue
    print("+"*20)
    print(jdx)
    print(u.draw())
    print(u.draw(output="latex_source"))
    optimized_qc=copy.deepcopy(u)
    check_old=str([optimized_qc.size(),optimized_qc.depth(),optimized_qc.count_ops()])
    print(check_old)
   
    for loop in range(1):
        print("loop:",loop)
        for idx,tc in enumerate(template_listN):

            if jdx==idx:
                continue
            pass_manager = PassManager()
            pass_manager.append(TemplateOptimization([tc],heuristics_qubits_param=None,
                heuristics_backward_param=None,
                user_cost_dict=None))
            optimized_qc = pass_manager.run(optimized_qc)
            check=str([optimized_qc.size(),optimized_qc.depth(),optimized_qc.count_ops()])
            if check!=check_old:
                #EffectiveOnes.append(tc)
                print("template index:",idx,check)
                #print(tc.draw())
                #QC=QuantumCircuit(tc.num_qubits)
                #QC.append(tc.to_gate(),[0,2,1])
                #print(QC.decompose().draw())
                #QC.append(u.to_gate(),[0,1,2])
                #print(QC.decompose().draw(output="latex_source"))
                print("optimized circuit")
                print(optimized_qc.draw())
            check_old=check

++++++++++++++++++++
0
   ┌───┐┌───┐
q: ┤ X ├┤ X ├
   └───┘└───┘
\documentclass[border=2px]{standalone}

\usepackage[braket, qm]{qcircuit}
\usepackage{graphicx}

\begin{document}
\scalebox{1.0}{
\Qcircuit @C=1.0em @R=0.2em @!R { \\
	 	\nghost{{q} :  } & \lstick{{q} :  } & \gate{\mathrm{X}} & \gate{\mathrm{X}} & \qw & \qw\\
\\ }}
\end{document}
[2, 2, OrderedDict({'x': 2})]
loop: 0
++++++++++++++++++++
1
               
q_0: ──■────■──
     ┌─┴─┐┌─┴─┐
q_1: ┤ X ├┤ X ├
     └───┘└───┘
\documentclass[border=2px]{standalone}

\usepackage[braket, qm]{qcircuit}
\usepackage{graphicx}

\begin{document}
\scalebox{1.0}{
\Qcircuit @C=1.0em @R=0.8em @!R { \\
	 	\nghost{{q}_{0} :  } & \lstick{{q}_{0} :  } & \ctrl{1} & \ctrl{1} & \qw & \qw\\
	 	\nghost{{q}_{1} :  } & \lstick{{q}_{1} :  } & \targ & \targ & \qw & \qw\\
\\ }}
\end{document}
[2, 2, OrderedDict({'cx': 2})]
loop: 0
++++++++++++++++++++
2
               
q_0: ──■────■──
       │    │  
q_1: ──■────■──
     ┌─┴─┐┌─┴─┐
q_2: ┤ X ├┤ X ├
     └─

In [13]:
from qiskit import qasm2
print(qasm2.dumps(optimized_qc))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
cx q[2],q[1];
ccx q[0],q[1],q[2];
ccx q[0],q[1],q[2];
cx q[2],q[1];


In [14]:
print(qasm2.loads("OPENQASM 2.0;\
include \"qelib1.inc\";\
qreg q[3];\
cx q[2],q[1];\
cx q[0],q[1];\
cx q[0],q[1];\
cx q[2],q[1];").draw(output="latex_source"))

\documentclass[border=2px]{standalone}

\usepackage[braket, qm]{qcircuit}
\usepackage{graphicx}

\begin{document}
\scalebox{1.0}{
\Qcircuit @C=1.0em @R=0.8em @!R { \\
	 	\nghost{{q}_{0} :  } & \lstick{{q}_{0} :  } & \qw & \ctrl{1} & \ctrl{1} & \qw & \qw & \qw\\
	 	\nghost{{q}_{1} :  } & \lstick{{q}_{1} :  } & \targ & \targ & \targ & \targ & \qw & \qw\\
	 	\nghost{{q}_{2} :  } & \lstick{{q}_{2} :  } & \ctrl{-1} & \qw & \qw & \ctrl{-1} & \qw & \qw\\
\\ }}
\end{document}


In [ ]:


###
###  As is done in the following lines,
###  we can execute the standard Template Optimization
###  using the pass manager.
###
#print()
##print("I optimize the circuit by TemplateOptimization using all nct templates.")
pass_manager = PassManager()
pass_manager.append(TemplateOptimization(template_listN,heuristics_qubits_param=None,
    heuristics_backward_param=None,
    user_cost_dict=None))
#optimized_qc = pass_manager.run(optimized_qc)
##print("The result is as follows.")
##print(optimized_qc.size(),optimized_qc.depth(),optimized_qc.count_ops())


###
### We can pick up the template that affects the circuit to be optimized.
### Those templates are stored in a list EffectiveOnes.
###
### But currently, we skip this step.
### EffectiveOnes is the set of all templates.
###
##print()
##print("I pick up the templates that potentially contribute to the optimization up #to a fixed point.")

for jdx,u in enumerate(template_listN):
    print("+"*20)
    print(jdx)
    print(u.draw())
    optimized_qc=copy.deepcopy(u)
    check_old=str([optimized_qc.size(),optimized_qc.depth(),optimized_qc.count_ops()])
    print(check_old)
   
    for loop in range(1):
        #print("loop:",loop)
        for idx,tc in enumerate(template_listN):
            if jdx==idx:
                continue
            pass_manager = PassManager()
            pass_manager.append(TemplateOptimization([tc],heuristics_qubits_param=None,
                heuristics_backward_param=None,
                user_cost_dict=None))
            optimized_qc = pass_manager.run(optimized_qc)
            check=str([optimized_qc.size(),optimized_qc.depth(),optimized_qc.count_ops()])
            if check!=check_old:
                #EffectiveOnes.append(tc)
                print("template index:",idx,check)
                print(tc.draw())
                print("optimized circuit")
                print(optimized_qc.draw())
            check_old=check
    

#print("I use the following template circuits that may actually works. I call them Effective Ones.")
#print(EffectiveOnes)

In [1]:
import json
import copy
from qiskit import qasm2
with open("./quartz/eccset/NCT_3_3_complete_ECC_set.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
def CIRCtoMONOM(v):

###Convert a circuit data to a Monomial

    if len(v)==0:
        return "1"
    A=[]    
    for u in v:
        w=u[1]
        m=u[0]
        for x in w:
            m+="_"+x
        A.append(m)
    mstr=""
    for u in A:
        mstr+=u+"*"
    return mstr[0:-1]    



def jsontoideal(data,n):
    A=[data[1][ky] for ky in data[1].keys()]

    MS=[]
    for u, v in A[n]:
        MS.append(CIRCtoMONOM(v))
    MS
    
    P=[]
    for u, v in A[n]:
        P.extend(CIRCtoMONOM(v).split("*"))
    P=list(set(P))
    P.sort()
    
    
    idealI="ideal I="
    for i in range(len(MS)):
        idealI+=MS[i]+"-"+MS[0]+","
    idealI=idealI[0:-1]
    #print(idealI)
    
    idealJ="ideal I1="
    for u in P:
        idealJ+=u+"*"+u+"-1,"
    idealJ=idealJ[0:-1]
    #idealJ
    
    
    rdef="ring r=0,("
    
    for u in P:
        rdef += u+","
    
    rdef=rdef[0:-1]
    rdef+="),dp;"
    rdef
    
    print(rdef)
    print("option(redSB);")
    print("def R = freeAlgebra(r, 15); setring R;")
    print(idealI+";")
    print(idealJ+";")
    print("letplaceGBasis(I+I1);")

for i in range(len(data[1])):
    jsontoideal(data,i)

ring r=0,(cx_Q0_Q1,x_Q0,x_Q1),dp;
option(redSB);
def R = freeAlgebra(r, 15); setring R;
ideal I=x_Q0*cx_Q0_Q1-x_Q0*cx_Q0_Q1,x_Q1*cx_Q0_Q1*x_Q0-x_Q0*cx_Q0_Q1;
ideal I1=cx_Q0_Q1*cx_Q0_Q1-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;
letplaceGBasis(I+I1);
ring r=0,(cx_Q1_Q0,x_Q0,x_Q1),dp;
option(redSB);
def R = freeAlgebra(r, 15); setring R;
ideal I=cx_Q1_Q0*x_Q1-cx_Q1_Q0*x_Q1,x_Q0*x_Q1*cx_Q1_Q0-cx_Q1_Q0*x_Q1;
ideal I1=cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;
letplaceGBasis(I+I1);
ring r=0,(cx_Q0_Q1,cx_Q1_Q0,x_Q0,x_Q1),dp;
option(redSB);
def R = freeAlgebra(r, 15); setring R;
ideal I=x_Q1*cx_Q1_Q0*cx_Q0_Q1-x_Q1*cx_Q1_Q0*cx_Q0_Q1,cx_Q1_Q0*cx_Q0_Q1*x_Q0-x_Q1*cx_Q1_Q0*cx_Q0_Q1;
ideal I1=cx_Q0_Q1*cx_Q0_Q1-1,cx_Q1_Q0*cx_Q1_Q0-1,x_Q0*x_Q0-1,x_Q1*x_Q1-1;
letplaceGBasis(I+I1);
ring r=0,(cx_Q0_Q1,cx_Q1_Q0),dp;
option(redSB);
def R = freeAlgebra(r, 15); setring R;
ideal I=cx_Q0_Q1*cx_Q1_Q0*cx_Q0_Q1-cx_Q0_Q1*cx_Q1_Q0*cx_Q0_Q1,cx_Q1_Q0*cx_Q0_Q1*cx_Q1_Q0-cx_Q0_Q1*cx_Q1_Q0*cx_Q0_Q1;
ideal I1=cx_Q0_Q1*cx_Q0_Q1-1,cx_Q1

In [8]:
def jsontoqasm(data,n):
    header="OPENQASM 2.0;\ninclude \"qelib1.inc\";"
    A=[data[1][ky] for ky in data[1].keys()]
    print("-"*100)
    print(f"A[{n}]=",A[n])
    print("\n")
    QB=[]
    for u, v in A[n]:
        #print(u,"|",v)
        for w in v:
            QB.extend(w[1])
            #print(w,"QB<=",QB)
    QB=list(set(QB))
    QB.sort()
    #print("QB:",QB)
    header+="\nqreg q["+str(len(QB))+"];"
    print(header)
    for u,v,in A[n]:
        print(v)
        qasmstr=copy.deepcopy(header)
        for w in v:
            #print("w:",w)
            qasmstr+=w[0]+" "+str(w[1]).replace("]","").replace("[","").replace("'","").replace("Q","q[").replace(",","],")+"];"
        print(qasmstr)
        qc=qasm2.loads(qasmstr)
        print(qc.draw())
        #print(qc.draw(output="latex_source"))
            
        
for i in range(len(data[1])):
    jsontoqasm(data,i)

----------------------------------------------------------------------------------------------------
A[0]= [[[2, 2], [['x', ['Q0'], ['Q0']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']]]], [[2, 3], [['x', ['Q1'], ['Q1']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']], ['x', ['Q0'], ['Q0']]]]]


OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
[['x', ['Q0'], ['Q0']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']]]
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];x q[0];cx q[0], q[1];
     ┌───┐     
q_0: ┤ X ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
[['x', ['Q1'], ['Q1']], ['cx', ['Q0', 'Q1'], ['Q0', 'Q1']], ['x', ['Q0'], ['Q0']]]
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];x q[1];cx q[0], q[1];x q[0];
               ┌───┐
q_0: ───────■──┤ X ├
     ┌───┐┌─┴─┐└───┘
q_1: ┤ X ├┤ X ├─────
     └───┘└───┘     
----------------------------------------------------------------------------------------------------
A[1]= [[[2, 2], [['cx', ['Q1', 'Q0'], ['Q1', 'Q0']], ['x', ['Q1'], ['Q1']]]], [[2, 3], [['x', ['Q0'], ['Q0